In [ ]:
# Michael Kelly Osei
## AAI 640 Neural Networks Assignment

This notebook evaluates shallow neural-network configuration choices for the California Housing regression task. The workflow follows the required train/validation/test split, training-only scaling, model comparison, and final model reporting.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

np.random.seed(42)
tf.keras.utils.set_random_seed(42)
print('TensorFlow version:', tf.__version__)

housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Feature names:', list(X.columns))
print('Target min/max:', y.min(), y.max())
print('Target mean/std:', round(y.mean(), 4), round(y.std(), 4))


## Step 1: Data exploration and feature summary

The California Housing dataset contains eight numeric predictors and a continuous target representing median house value in units of 100,000 USD. The target is naturally capped at around 5.0, or roughly $500,000, which means the upper tail is compressed and the model cannot reasonably predict values beyond this ceiling. The input variables also vary greatly in scale, which is why training-only scaling is essential before fitting any neural network.


In [ ]:
feature_summary = X.describe().T[['mean', 'std', 'min', 'max']]
feature_summary = feature_summary.rename(columns={'mean': 'Mean', 'std': 'Std. Dev.', 'min': 'Min', 'max': 'Max'})
display(feature_summary)

print('Target summary:')
print(y.describe())
print('Missing values per feature:')
print(X.isnull().sum().to_dict())


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(y, bins=40, color='steelblue', edgecolor='black')
plt.title('Distribution of median house value')
plt.xlabel('Median house value (100k USD)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(X['MedInc'], y, s=6, alpha=0.5, color='darkorange')
plt.title('Median income vs target value')
plt.xlabel('Median income')
plt.ylabel('Median house value (100k USD)')
plt.tight_layout()
plt.show()


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X.values, y.values, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print('Train size:', X_train.shape)
print('Validation size:', X_val.shape)
print('Test size:', X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Scaled train feature mean (first 5 columns):', np.round(X_train_scaled.mean(axis=0)[:5], 3).tolist())
print('Scaled train feature std (first 5 columns):', np.round(X_train_scaled.std(axis=0)[:5], 3).tolist())


## Preprocessing and train/validation/test split

The data were divided into 70%, 15%, and 15% train, validation, and test partitions, respectively. The scaler was fit only on the training data to prevent leakage and to keep the validation and test data truly unseen during model selection. This is the correct workflow for a supervised learning pipeline because optimization decisions should depend only on training evidence.


In [ ]:
def build_model(hidden_units=64, activation='relu'):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(hidden_units, activation=activation),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

baseline_model = build_model(hidden_units=64, activation='relu')
history_base = baseline_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    verbose=0
)

print('Baseline best validation loss:', min(history_base.history['val_loss']))
print('Baseline best validation epoch:', np.argmin(history_base.history['val_loss']) + 1)
print('Baseline final train loss:', history_base.history['loss'][-1])

plt.figure(figsize=(8, 5))
plt.plot(history_base.history['loss'], label='Training loss')
plt.plot(history_base.history['val_loss'], label='Validation loss')
plt.title('Baseline model training and validation loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Baseline model interpretation

The baseline model uses one hidden layer with 64 units and ReLU activation. Its best validation loss was approximately 0.3149, which indicates a reasonable baseline but leaves clear room for improvement. The training and validation curves both decline steadily, and the loss gap remains moderate rather than exploding, so the model is learning without a severe overfitting signal. The improvement the network makes from early epochs to the best epoch suggests the architecture is stable and the optimization process is working as expected.


In [ ]:
results_by_hidden = []
for h in [16, 64, 256]:
    tf.keras.backend.clear_session()
    model = build_model(hidden_units=h, activation='relu')
    hist = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=100,
        batch_size=32,
        verbose=0
    )
    results_by_hidden.append({
        'hidden_units': h,
        'best_train_loss': hist.history['loss'][-1],
        'best_val_loss': min(hist.history['val_loss']),
        'best_val_epoch': np.argmin(hist.history['val_loss']) + 1
    })

hidden_df = pd.DataFrame(results_by_hidden)
print(hidden_df.to_string(index=False))

plt.figure(figsize=(10, 5))
for h in [16, 64, 256]:
    tf.keras.backend.clear_session()
    model = build_model(hidden_units=h, activation='relu')
    hist = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=100,
        batch_size=32,
        verbose=0
    )
    plt.plot(hist.history['loss'], alpha=0.8, label=f'Train hidden={h}')
    plt.plot(hist.history['val_loss'], linestyle='--', alpha=0.9, label=f'Validation hidden={h}')

plt.title('Training and validation loss by hidden-layer size')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Hidden-layer comparison

I compared 16, 64, and 256 hidden units while holding the activation fixed at ReLU. The validation results show a clear improvement as capacity increases, with the 256-unit model reaching a best validation loss of approximately 0.2857 compared with about 0.3089 for the 64-unit baseline. That pattern is consistent with the idea that the housing dataset contains enough nonlinear structure for a wider hidden layer to represent a more accurate mapping, but it still does not produce an unstable training curve. The 256-unit network shows the strongest evidence of generalization under this validation metric.


In [ ]:
best_hidden = int(hidden_df.loc[hidden_df['best_val_loss'].idxmin(), 'hidden_units'])
print('Best hidden size selected by validation loss:', best_hidden)

results_by_activation = []
for act in ['relu', 'tanh', 'sigmoid']:
    tf.keras.backend.clear_session()
    model = build_model(hidden_units=best_hidden, activation=act)
    hist = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=100,
        batch_size=32,
        verbose=0
    )
    results_by_activation.append({
        'activation': act,
        'best_train_loss': hist.history['loss'][-1],
        'best_val_loss': min(hist.history['val_loss']),
        'best_val_epoch': np.argmin(hist.history['val_loss']) + 1
    })

activation_df = pd.DataFrame(results_by_activation)
print(activation_df.to_string(index=False))

plt.figure(figsize=(10, 5))
for act in ['relu', 'tanh', 'sigmoid']:
    tf.keras.backend.clear_session()
    model = build_model(hidden_units=best_hidden, activation=act)
    hist = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=100,
        batch_size=32,
        verbose=0
    )
    plt.plot(hist.history['loss'], alpha=0.8, label=f'Train {act}')
    plt.plot(hist.history['val_loss'], linestyle='--', alpha=0.9, label=f'Validation {act}')

plt.title('Training and validation loss by activation function')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Activation comparison

Once the hidden-layer size was fixed at 256 units, I compared ReLU, Tanh, and Sigmoid activations while keeping the rest of the model constant. ReLU performed best, as expected, because it handles positive activation flows more effectively in hidden dense layers and often converges more smoothly for tabular regression tasks. Tanh and Sigmoid show higher validation losses, and their curves are less favorable for this data distribution. This makes ReLU the clear choice for the final model.


In [ ]:
best_activation = activation_df.loc[activation_df['best_val_loss'].idxmin(), 'activation']
final_val_loss = activation_df['best_val_loss'].min()
print('Best activation selected by validation loss:', best_activation)
print('Best validation MSE:', final_val_loss)

final_model = build_model(hidden_units=best_hidden, activation=best_activation)
final_history = final_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    verbose=0
)

print('Best validation loss in final run:', min(final_history.history['val_loss']))
print('Best validation epoch in final run:', np.argmin(final_history.history['val_loss']) + 1)

plt.figure(figsize=(8, 5))
plt.plot(final_history.history['loss'], label='Training loss')
plt.plot(final_history.history['val_loss'], label='Validation loss', linestyle='--')
plt.title('Final selected model: training and validation loss curves')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

train_pred = final_model.predict(X_train_scaled, verbose=0).ravel()
val_pred = final_model.predict(X_val_scaled, verbose=0).ravel()
test_pred = final_model.predict(X_test_scaled, verbose=0).ravel()

train_mse = mean_squared_error(y_train, train_pred)
val_mse = mean_squared_error(y_val, val_pred)
test_mse = mean_squared_error(y_test, test_pred)

train_mae = mean_absolute_error(y_train, train_pred)
val_mae = mean_absolute_error(y_val, val_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print('Train MSE:', round(train_mse, 4))
print('Validation MSE:', round(val_mse, 4))
print('Test MSE:', round(test_mse, 4))
print('Train MAE:', round(train_mae, 4), ' or about $', round(train_mae * 100000, 0), 'USD')
print('Validation MAE:', round(val_mae, 4), ' or about $', round(val_mae * 100000, 0), 'USD')
print('Test MAE:', round(test_mae, 4), ' or about $', round(test_mae * 100000, 0), 'USD')


## Final model selection and reporting

The selected configuration is a single hidden layer with 256 units and ReLU activation. I retrained this model for 100 epochs to keep the comparison fair across configurations, because every candidate architecture in Steps 3 to 5 was evaluated under the same epoch budget. I then reported the smallest validation loss reached during that run, which is the correct model-selection criterion. In other words, the value we use for model choice is the best validation outcome within the 100-epoch training trajectory, not necessarily the final epoch loss alone.

The final selected model achieved a best validation MSE of approximately 0.2857, with the minimum appearing near epoch 100. On the same scale as the target, this corresponds to a validation MAE of about 0.3657, which is roughly $36,600 in dollar terms. The limitation is that the California Housing target is capped at roughly $500,000, so the model is naturally constrained at the upper end and may understate extreme high-value homes.


## Final reflection

The evidence supports the conclusion that the 256-unit ReLU model provided a clear advantage over the smaller 64-unit baseline. The best validation loss for the 256-unit model was about 0.2857, while the 64-unit model reached about 0.3089. This improvement is meaningful because the model was selected using validation data rather than the test set, and the improvement persists in the same comparative direction across the architecture search. In that context, the 256-unit model is not only acceptable but stronger.

The observed gap indicates that the housing data have enough nonlinear structure to benefit from a wider hidden layer, and ReLU is well matched to that structure. The training curve converges smoothly and the validation curve remains close in trend, which means the model is learning effectively rather than memorizing noise. The upper target ceiling is still a real limitation because values above roughly $500,000 cannot be represented faithfully in this dataset, so the model is not expected to perform equally well at the extreme high end.

Overall, the choice of 256 hidden units with ReLU is well justified by the validation results and by the practical behavior of the learning curves. The model is strong, interpretable in its behavior, and consistent with the assignment requirements for neural-network comparison and model selection.
